# 13) Explore Risk Categories via Clustering (Small-scale)

目的：**事前定義カテゴリ（サイバー/地政学/人材）以外に、データ上どんなカテゴリが多いか**を探索します。

このノートブックでやること（小規模）：
- DBからリスク文をサンプル
- 前処理（既存の簡易 normalize）
- チャンク化（段落/箇条書き）
- 埋め込み（SBERT: `pkshatech/GLuCoSE-base-ja`）
- KMeansでクラスタリング
- 各クラスタの **代表チャンク（例：中心に近いもの）** と **キーワード（TF-IDF）** を出力

アウトプット（スライド/論文用の材料）：
- クラスタ上位K（例：5〜10）のラベル案と代表例

> 注意：これは探索（exploratory）なので、主結果ではなく「カテゴリ候補の発見・妥当性補強」に使うのが安全です。



In [ ]:
import os
import re
import unicodedata

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

# optional plotting (works in notebook env)
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)



In [ ]:
def find_project_root(start_dir: str) -> str:
    d = os.path.abspath(start_dir)
    while True:
        if os.path.exists(os.path.join(d, "docker-compose.yml")) and os.path.exists(os.path.join(d, "requirements.txt")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(start_dir)
        d = parent


PROJECT_ROOT = find_project_root(os.getcwd())
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

load_dotenv(dotenv_path=os.path.join(PROJECT_ROOT, ".env"))
DB_USER = os.getenv("POSTGRES_USER")
DB_PASS = os.getenv("POSTGRES_PASSWORD")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("DB connected")

# --- small-scale config ---
SAMPLE_DOCS = 400
SAMPLE_SEED = 42
MAX_CHUNKS = 5000

MODEL_NAME = os.getenv("RISK_EMBED_MODEL", "pkshatech/GLuCoSE-base-ja")
BATCH_SIZE = 32

# number of clusters to try
K = 12



In [ ]:
HEADER_PATTERNS = [
    r"【.*?】",
    r"^\s*[０-９0-9]+\s*",
    r"^\s*第[０-９0-9]+",
]

BOILERPLATE_PATTERNS = [
    r"有価証券報告書に記載した.*?認識している主要なリスクは、以下のとおり.*?。",
    r"なお、文中の将来に関する事項は.*?判断したものであります。",
    r"文中の将来に関する事項は.*?判断したものであります。",
    r"当社グループは、これらのリスクを認識したうえで.*?努めてまいります。",
    r"詳細については、.*?をご参照ください。",
]


def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)

    lines = []
    for line in s.splitlines():
        t = line
        for p in HEADER_PATTERNS:
            t = re.sub(p, "", t)
        t = t.strip()
        if t:
            lines.append(t)
    s = "\n".join(lines)

    s = unicodedata.normalize("NFKC", s)

    for p in BOILERPLATE_PATTERNS:
        s = re.sub(p, "", s, flags=re.DOTALL)

    s = re.sub(r"[ \t\u3000]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s).strip()

    # numeric masks
    s = re.sub(r"\d{1,3}(,\d{3})+\s*円", "<MONEY>", s)
    s = re.sub(r"\d+(\.\d+)?\s*%", "<PCT>", s)
    s = re.sub(r"(19|20)\d{2}年", "<YEAR>", s)
    s = re.sub(r"(19|20)\d{2}", "<YEAR>", s)
    s = re.sub(r"\d+(\.\d+)?", "<NUM>", s)

    s = s.replace("当社グループ", "<COMPANY>").replace("当社", "<COMPANY>")
    s = s.replace("可能性があります", "<MODAL>").replace("おそれがあります", "<MODAL>").replace("恐れがあります", "<MODAL>")

    s = re.sub(r"\s+", " ", s).strip()
    return s


BULLET_SPLIT_RE = re.compile(r"(?:\n|^)(?:[・●]|\(?[0-9]+\)?|[①②③④⑤⑥⑦⑧⑨⑩]|[a-zA-Z]\)|\([ivxIVX]+\))\s*")


def chunk_text(text: str, min_len: int = 30, target_min: int = 80, target_max: int = 350) -> list[str]:
    if not text:
        return []

    parts = [p.strip() for p in text.split("\n") if p.strip()]
    finer = []
    for p in parts:
        x = BULLET_SPLIT_RE.sub("\n", p)
        finer.extend([q.strip() for q in x.split("\n") if q.strip()])

    chunks = []
    for p in finer:
        if len(p) <= target_max:
            chunks.append(p)
            continue

        sents = [s.strip() for s in p.split("。") if s.strip()]
        buf = []
        for s in sents:
            buf.append(s + "。")
            if len(buf) >= 2:
                chunks.append("".join(buf).strip())
                buf = []
        if buf:
            chunks.append("".join(buf).strip())

    merged = []
    for c in chunks:
        if not merged:
            merged.append(c)
            continue
        if len(c) < min_len:
            merged[-1] = (merged[-1] + " " + c).strip()
        else:
            merged.append(c)

    final = []
    for c in merged:
        if not final:
            final.append(c)
            continue
        if len(c) < target_min:
            final[-1] = (final[-1] + " " + c).strip()
        else:
            final.append(c)

    return [c for c in final if c.strip()]



In [ ]:
# --- 1) Load sample documents ---

docs = pd.read_sql(
    text(
        """
        select doc_id, company_id::uuid as company_id, fiscal_year, risk_text
        from edinet_documents
        where risk_text is not null and risk_text != ''
        order by random()
        limit :n
        """
    ),
    engine,
    params={"n": int(SAMPLE_DOCS)},
)
print("docs:", len(docs))

# preprocess
_docs = docs.copy()
_docs["risk_text_norm"] = _docs["risk_text"].map(normalize_text)
_docs["n_chars"] = _docs["risk_text_norm"].str.len()
_docs = _docs[_docs["n_chars"] >= 200].copy()
_docs = _docs.sample(n=min(len(_docs), SAMPLE_DOCS), random_state=SAMPLE_SEED).reset_index(drop=True)
print("docs after filter:", len(_docs), "median chars:", int(_docs["n_chars"].median()))

_docs[["doc_id", "fiscal_year", "n_chars"]].head()



In [ ]:
# --- 2) Chunk documents ---

rows = []
for r in _docs.itertuples(index=False):
    chunks = chunk_text(r.risk_text_norm)
    for c in chunks:
        rows.append(
            {
                "doc_id": r.doc_id,
                "company_id": r.company_id,
                "fiscal_year": int(r.fiscal_year),
                "chunk": c,
                "chunk_len": int(len(c)),
            }
        )

chunks = pd.DataFrame(rows)
print("chunks:", len(chunks), "unique docs:", chunks["doc_id"].nunique())

# cap for speed
chunks = chunks.sort_values("chunk_len", ascending=False).head(MAX_CHUNKS).reset_index(drop=True)
print("chunks capped:", len(chunks), "median len:", int(chunks["chunk_len"].median()))

chunks.head(3)



In [ ]:
# --- 3) Embed chunks ---

print("MODEL_NAME:", MODEL_NAME)
model = SentenceTransformer(MODEL_NAME)

emb = model.encode(
    chunks["chunk"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
)
print("emb:", emb.shape)



In [ ]:
# --- 4) Clustering (KMeans) ---

kmeans = KMeans(n_clusters=int(K), random_state=42, n_init="auto")
labels = kmeans.fit_predict(emb)
chunks["cluster"] = labels

sizes = chunks["cluster"].value_counts().sort_values(ascending=False)
print("cluster sizes:\n", sizes)

# quick bar plot
ax = sizes.plot(kind="bar")
ax.set_title(f"Cluster sizes (K={K})")
ax.set_xlabel("cluster")
ax.set_ylabel("#chunks")
plt.tight_layout()
plt.show()



In [ ]:
# --- 5) Interpret clusters: keywords + representative chunks ---

# TF-IDF for cluster keywords
vec = TfidfVectorizer(
    token_pattern=r"(?u)\\b\\w+\\b",
    min_df=3,
    max_df=0.5,
)
X = vec.fit_transform(chunks["chunk"].tolist())
terms = np.array(vec.get_feature_names_out())

centers = kmeans.cluster_centers_

summary_rows = []
for c in sizes.index.tolist():
    idx = np.where(chunks["cluster"].to_numpy() == c)[0]
    if len(idx) == 0:
        continue

    # keywords: mean tfidf within cluster
    tfidf_mean = np.asarray(X[idx].mean(axis=0)).ravel()
    top_terms = terms[np.argsort(tfidf_mean)[-12:]][::-1]

    # representative examples: closest to center
    d = 1.0 - (emb[idx] @ centers[c])  # cosine distance since emb normalized
    ex_idx = idx[np.argsort(d)[:5]]

    examples = chunks.loc[ex_idx, ["doc_id", "fiscal_year", "chunk"]].to_dict("records")

    summary_rows.append(
        {
            "cluster": int(c),
            "n_chunks": int(len(idx)),
            "keywords": " ".join(top_terms.tolist()),
            "examples": examples,
        }
    )

summary = pd.DataFrame(summary_rows)
summary = summary.sort_values("n_chunks", ascending=False).reset_index(drop=True)

# Pretty print top clusters
TOP_SHOW = min(8, len(summary))
for r in summary.head(TOP_SHOW).itertuples(index=False):
    print("\n=== cluster", r.cluster, "n=", r.n_chunks, "===")
    print("keywords:", r.keywords)
    for j, ex in enumerate(r.examples, start=1):
        s = ex["chunk"]
        print(f"  [{j}] doc={ex['doc_id']} fy={ex['fiscal_year']} :: {s[:180]}")

summary[["cluster", "n_chunks", "keywords"]].head(TOP_SHOW)



In [ ]:
# --- 6) Export cluster summary (for later labeling) ---

OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "results")
os.makedirs(OUT_DIR, exist_ok=True)

# Flatten examples for CSV readability (keep first 3)
flat = summary.copy()
flat["ex1"] = flat["examples"].apply(lambda xs: xs[0]["chunk"][:200] if xs else "")
flat["ex2"] = flat["examples"].apply(lambda xs: xs[1]["chunk"][:200] if len(xs) > 1 else "")
flat["ex3"] = flat["examples"].apply(lambda xs: xs[2]["chunk"][:200] if len(xs) > 2 else "")
flat = flat.drop(columns=["examples"])

out_csv = os.path.join(OUT_DIR, f"cluster_summary_k{K}_n{len(chunks)}.csv")
flat.to_csv(out_csv, index=False)
print("wrote:", out_csv)

flat.head(10)

